In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip /content/drive/MyDrive/Colab\ Notebooks/train_xe.zip -d /content/dataset

Archive:  /content/drive/MyDrive/Colab Notebooks/train_xe.zip
 extracting: /content/dataset/README.dataset.txt  
 extracting: /content/dataset/README.roboflow.txt  
 extracting: /content/dataset/data.yaml  
   creating: /content/dataset/test/
   creating: /content/dataset/test/images/
 extracting: /content/dataset/test/images/IMG_0377_2-0002_jpg.rf.86a4b42e66ca299943998387f569036f.jpg  
 extracting: /content/dataset/test/images/IMG_0377_2-0003_jpg.rf.5e42911373e12caeaeef499a4413e3ea.jpg  
 extracting: /content/dataset/test/images/IMG_0377_2-0007_jpg.rf.3f8c74c6d996fa48b79536b5cd406974.jpg  
 extracting: /content/dataset/test/images/IMG_0377_2-0094_jpg.rf.bd259a7d5ba8f9c6b3930e57907bdc60.jpg  
 extracting: /content/dataset/test/images/IMG_0377_2-0119_jpg.rf.01467b96ff0de62b4dfaa4d012f57f23.jpg  
 extracting: /content/dataset/test/images/IMG_0377_2-0142_jpg.rf.c59d54771a17217f06b94e61ff80f20f.jpg  
 extracting: /content/dataset/test/images/IMG_0377_2-0192_jpg.rf.6d0442a1bb00bc7c75c7daf80

In [5]:
import yaml

data_yaml_content = '''
path: /content/dataset
train: train/images
val: valid/images
test: test/images
nc: 1
names: ["vehicle"]
'''

# Write the YAML content to a file
with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print('data.yaml successfully created in /content/dataset/')


data.yaml successfully created in /content/dataset/


In [11]:
!yolo detect train data=/content/dataset/data.yaml model=yolov10n.pt epochs=50 imgsz=640

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

In [10]:
import os

def process_label_file(filepath):
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if parts:
                # Change the first part (class ID) to '0'
                parts[0] = '0'
                new_lines.append(' '.join(parts) + '\n')
            else:
                new_lines.append('\n') # Keep empty lines

        with open(filepath, 'w') as f:
            f.writelines(new_lines)
        # print(f"Processed: {filepath}")
    except Exception as e:
        print(f"Error processing {filepath}: {e}")

# Define the base dataset path
dataset_base_path = '/content/dataset'

# Define paths for train and valid labels
train_labels_path = os.path.join(dataset_base_path, 'train', 'labels')
valid_labels_path = os.path.join(dataset_base_path, 'valid', 'labels')

# Process train labels
print(f"Processing labels in {train_labels_path}...")
for filename in os.listdir(train_labels_path):
    if filename.endswith('.txt'):
        process_label_file(os.path.join(train_labels_path, filename))

# Process valid labels
print(f"Processing labels in {valid_labels_path}...")
for filename in os.listdir(valid_labels_path):
    if filename.endswith('.txt'):
        process_label_file(os.path.join(valid_labels_path, filename))

print("Label files processing complete. All class IDs should now be '0'.")


Processing labels in /content/dataset/train/labels...
Processing labels in /content/dataset/valid/labels...
Label files processing complete. All class IDs should now be '0'.


In [14]:
!yolo export model=runs/detect/train2/weights/best.pt format=onnx

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv10n summary (fused): 102 layers, 2,265,363 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from 'runs/detect/train2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 161ms
Prepared 4 packages in 1.39s
Installed 4 packages in 248ms
 + colorama==0.4.6
 + onnx==1.20.1
 + onnxruntime==1.24.3
 + onnxslim==0.1.88

requirements: AutoUpdate success ✅ 2.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...
/usr/local/lib/python3.12

In [15]:
from google.colab import files

# Path to your best.pt file
model_path = '/content/runs/detect/train2/weights/best.pt'

# Download the file
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>